# BERT-Base-Uncased Transformer Ranker - Smart MCQ Solver (With Wandb Epoch Loss Curves)

### Overview and Wandb Tracking Features
This notebook implements a fine-tuned **BERT-Base-Uncased (`bert-base-uncased`) Transformer Ranker** for the **Smart MCQ Solver Challenge** with **Epoch-Wise Loss Curves, Average Loss Metrics, and Single Train/Validation Split**.

- **Primary Kaggle Metric**: MAP@3 (Mean Average Precision at 3)
- **Secondary Evaluation Metrics**:
  - Top-1 Accuracy: Standard classification accuracy (single best prediction).
  - Top-3 Accuracy: Recall metric checking if ground truth is within top 3 ranked answers.
  - F1-Score (Macro): Harmonic mean of precision and recall.
  - Precision and Recall (Macro)
  - Log Loss: Binary Cross Entropy of option probability predictions.

- **Wandb Dashboard Loss Charts and Visualizations**:
  - Epoch-Wise Average Training Loss (`epoch/avg_train_loss`)
  - Epoch-Wise Average Validation Loss (`epoch/avg_val_loss`)
  - Epoch-Wise MAP@3 Score Curve (`epoch/val_map3`)
  - Automatic Matplotlib Loss & Metric Progression Curves.

---


In [ ]:
# Cell 1: Core Imports and Dependencies
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, log_loss
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Core libraries imported successfully! Device: {device}")


In [ ]:
# Cell 2: Weights and Biases (Wandb) Setup
try:
    import wandb
    WANDB_API_KEY = "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"
    WANDB_ENTITY = "models-indian-institute-of-technology-madras481" 
    WANDB_PROJECT = "23f3001092-t22026"

    wandb.login(key=WANDB_API_KEY)
    USE_WANDB = True
    print("Wandb initialized successfully!")
except Exception as e:
    USE_WANDB = False
    print(f"Wandb running in offline fallback mode ({e})")


In [ ]:
# Cell 3: Dataset Loading and Preprocessing
def load_datasets():
    train_candidates = [
        "train.csv",
        "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
        "/kaggle/input/smart-mcq-solver-challenge/train.csv"
    ]
    test_candidates = [
        "test.csv",
        "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv",
        "/kaggle/input/smart-mcq-solver-challenge/test.csv"
    ]
    
    train_path = next((p for p in train_candidates if os.path.exists(p)), train_candidates[0])
    test_path = next((p for p in test_candidates if os.path.exists(p)), test_candidates[0])
    
    df_train = pd.read_csv(train_path).fillna('')
    df_test = pd.read_csv(test_path).fillna('')
    
    return df_train, df_test

train_df, test_df = load_datasets()
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']

print(f"Train shape: {train_df.shape}")
print(f"Test shape : {test_df.shape}")
train_df.head(3)


In [ ]:
# Cell 4: Evaluation Metric Functions
def apk(actual, predicted, k=3):
    if actual in predicted[:k]:
        return 1.0 / (predicted[:k].index(actual) + 1.0)
    return 0.0

def mapk(actuals, preds, k=3):
    return float(np.mean([apk(a, p, k) for a, p in zip(actuals, preds)]))

def compute_all_metrics(actuals, pred_lists):
    map3_score = mapk(actuals, pred_lists)
    top1_preds = [p[0] for p in pred_lists]
    top1_acc = accuracy_score(actuals, top1_preds)
    top3_acc = np.mean([1 if a in p[:3] else 0 for a, p in zip(actuals, pred_lists)])
    prec, rec, f1, _ = precision_recall_fscore_support(actuals, top1_preds, average='macro', zero_division=0)
    
    return {
        'map3': map3_score,
        'top1_accuracy': top1_acc,
        'top3_accuracy': top3_acc,
        'f1_score': f1,
        'precision': prec,
        'recall': rec
    }

def rank_preds(pw, scores, orig_df):
    pw_temp = pw.copy()
    pw_temp['score'] = scores
    
    ranked = pw_temp.sort_values(['qid', 'score'], ascending=[True, False])
    grouped = ranked.groupby('qid')['option'].apply(lambda opts: list(opts)[:3]).to_dict()
    
    return [grouped.get(qid, ['A', 'B', 'C']) for qid in orig_df['id']]


In [ ]:
# Cell 5: Pairwise Text Formatting and Tokenization
def make_pairwise_df(df, is_train=True):
    rows = []
    for _, r in df.iterrows():
        qid = r['id']
        prompt = str(r['prompt'])
        ans = r.get('answer', None)
        for opt in OPTION_COLS:
            opt_str = str(r[opt])
            combined = f"Question: {prompt} [SEP] Option: {opt_str}"
            row = {'qid': qid, 'option': opt, 'text': combined}
            if is_train:
                row['label'] = 1 if (ans == opt) else 0
            rows.append(row)
    return pd.DataFrame(rows)

pair_train = make_pairwise_df(train_df, is_train=True)
pair_test = make_pairwise_df(test_df, is_train=False)

MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Pairwise Train Shape: {pair_train.shape}")
print(f"Pairwise Test Shape : {pair_test.shape}")


In [ ]:
# Cell 6: PyTorch BERT Ranker Model Architecture
class MCQDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=256, is_train=True):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = str(self.data.iloc[idx]["text"])
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }
        
        if self.is_train and "label" in self.data.columns:
            item["labels"] = torch.tensor(self.data.iloc[idx]["label"], dtype=torch.float32)
            
        return item

class BertRanker(nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(cls_output)
        logits = self.fc(pooled).squeeze(-1)
        return logits

print("BertRanker model architecture loaded.")


In [ ]:
# Cell 7: Single Train/Val Split BERT Training with Epoch-Wise Loss Curves and Wandb Logging
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(pair_train, pair_train['label'], pair_train['qid']))

train_sub = pair_train.iloc[train_idx].reset_index(drop=True)
val_sub = pair_train.iloc[val_idx].reset_index(drop=True)

EPOCHS = 3 if torch.cuda.is_available() else 1
BATCH_SIZE = 16 if torch.cuda.is_available() else 4
LR = 2e-5

config_dict = {
    "architecture": "BERT-Base-Uncased Transformer Ranker",
    "dataset": "Smart MCQ Solver Challenge",
    "model_name": MODEL_NAME,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "epochs": EPOCHS,
    "validation_split": "Single Train/Val Split (80/20)"
}

if USE_WANDB:
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name="bert-base-uncased-single-split",
        config=config_dict
    )

print(f"\nStarting BERT-Base Fine-Tuning (Epochs: {EPOCHS}, Device: {device})...")

history_train_loss = []
history_val_loss = []
history_val_map3 = []

train_ds = MCQDataset(train_sub, tokenizer, max_len=MAX_LEN, is_train=True)
val_ds = MCQDataset(val_sub, tokenizer, max_len=MAX_LEN, is_train=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False)

# Fix: load on CPU first then move to GPU — prevents AcceleratorError
model = BertRanker(MODEL_NAME)
model = model.to(device)
if torch.cuda.is_available():
    torch.backends.cudnn.allow_tf32 = False  # prevents kernel mismatch on some Kaggle GPUs
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)
criterion = nn.BCEWithLogitsLoss()

best_val_loss = float('inf')
best_val_probs = None
best_model_state = None

for epoch in range(EPOCHS):
    # 1. Training Phase
    model.train()
    running_train_loss = 0.0
    total_train_batches = 0
    
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        
        logits_clean = torch.nan_to_num(logits, nan=0.0)
        loss = criterion(logits_clean, labels)
        
        if not torch.isnan(loss):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            running_train_loss += loss.item()
            total_train_batches += 1
            
    avg_train_loss = running_train_loss / max(total_train_batches, 1)
    
    # 2. Validation Phase
    model.eval()
    val_probs_epoch = []
    running_val_loss = 0.0
    total_val_batches = 0
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids, attention_mask)
            logits_clean = torch.nan_to_num(logits, nan=0.0)
            v_loss = criterion(logits_clean, labels)
            probs = torch.sigmoid(logits_clean)
            
            running_val_loss += v_loss.item()
            total_val_batches += 1
            val_probs_epoch.extend(probs.cpu().numpy())
            
    avg_val_loss = running_val_loss / max(total_val_batches, 1)
    val_probs_epoch = np.array(val_probs_epoch)
    
    val_qids = val_sub['qid'].unique()
    val_df_orig = train_df[train_df['id'].isin(val_qids)].reset_index(drop=True)
    
    epoch_preds = rank_preds(val_sub, val_probs_epoch, val_df_orig)
    epoch_actuals = val_df_orig['answer'].tolist()
    m_epoch = compute_all_metrics(epoch_actuals, epoch_preds)
    
    history_train_loss.append(avg_train_loss)
    history_val_loss.append(avg_val_loss)
    history_val_map3.append(m_epoch['map3'])
    
    print(f" Epoch {epoch+1}/{EPOCHS} | Avg Train Loss: {avg_train_loss:.4f} | Avg Val Loss: {avg_val_loss:.4f} | Val MAP@3: {m_epoch['map3']:.4f}")
    
    if USE_WANDB:
        wandb.log({
            "epoch": epoch + 1,
            "epoch/avg_train_loss": avg_train_loss,
            "epoch/avg_val_loss": avg_val_loss,
            "epoch/val_map3": m_epoch['map3'],
            "epoch/val_top1_accuracy": m_epoch['top1_accuracy'],
            "epoch/val_top3_accuracy": m_epoch['top3_accuracy'],
            "epoch/val_f1_score": m_epoch['f1_score']
        })
        
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_val_probs = val_probs_epoch
        best_model_state = model.state_dict().copy()

if best_model_state is not None:
    model.load_state_dict(best_model_state)

# Test Inference
test_ds = MCQDataset(pair_test, tokenizer, max_len=MAX_LEN, is_train=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False)

fold_test_probs = []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        logits = model(input_ids, attention_mask)
        logits_clean = torch.nan_to_num(logits, nan=0.0)
        probs = torch.sigmoid(logits_clean)
        fold_test_probs.extend(probs.cpu().numpy())

tst_bert = np.array(fold_test_probs)

val_qids = val_sub['qid'].unique()
val_df_orig = train_df[train_df['id'].isin(val_qids)].reset_index(drop=True)
m_bert = compute_all_metrics(val_df_orig['answer'].tolist(), rank_preds(val_sub, best_val_probs, val_df_orig))
clipped_val = np.clip(best_val_probs, 1e-15, 1 - 1e-15)
val_loss_score = log_loss(val_sub['label'], clipped_val)

print("\n==================================================")
print("       VALIDATION SET PERFORMANCE SUMMARY")
print("==================================================")
print(f" Primary Kaggle Metric (MAP@3): {m_bert['map3']:.4f}")
print(f" Top-1 Accuracy               : {m_bert['top1_accuracy']:.4f}")
print(f" Top-3 Recall Accuracy        : {m_bert['top3_accuracy']:.4f}")
print(f" Macro F1 Score               : {m_bert['f1_score']:.4f}")
print(f" Macro Precision              : {m_bert['precision']:.4f}")
print(f" Macro Recall                 : {m_bert['recall']:.4f}")
print(f" Pairwise Log Loss            : {val_loss_score:.4f}")
print("==================================================\n")

if USE_WANDB:
    wandb.log({
        "val/overall_map3": m_bert['map3'],
        "val/overall_top1_accuracy": m_bert['top1_accuracy'],
        "val/overall_top3_accuracy": m_bert['top3_accuracy'],
        "val/overall_f1_score": m_bert['f1_score'],
        "val/overall_precision": m_bert['precision'],
        "val/overall_recall": m_bert['recall'],
        "val/overall_val_loss": val_loss_score
    })
    
    table = wandb.Table(columns=["Model Architecture", "MAP@3 Score", "Top-1 Accuracy", "Top-3 Accuracy", "F1 Score", "Precision", "Recall", "Log Loss"])
    table.add_data("BERT-Base-Uncased Transformer Ranker", m_bert['map3'], m_bert['top1_accuracy'], m_bert['top3_accuracy'], m_bert['f1_score'], m_bert['precision'], m_bert['recall'], val_loss_score)
    
    wandb.log({"model_comparative_summary": table})
    wandb.finish()


In [ ]:
# Cell 8: Plot Epoch Loss Curves and Comparative Report Table
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_train_loss, label='Avg Train Loss', color='blue', linewidth=2)
plt.plot(history_val_loss, label='Avg Val Loss', color='orange', linewidth=2)
plt.title('BERT-Base-Uncased Training vs Validation Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('BCE Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_val_map3, label='Validation MAP@3', color='green', linewidth=2)
plt.title('BERT-Base-Uncased Validation MAP@3 Progression')
plt.xlabel('Epochs')
plt.ylabel('MAP@3 Score')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('bert_loss_curves.png')
plt.show()

test_preds = rank_preds(pair_test, tst_bert, test_df)

submission = pd.DataFrame({
    'ID': test_df['id'],
    'Prediction': [' '.join(p) for p in test_preds]
})

submission.to_csv('submission.csv', index=False)
print("Successfully generated submission.csv!")

assert len(submission) == len(test_df), f"Expected {len(test_df)} rows, got {len(submission)}"
assert submission['Prediction'].str.match(r'^[A-E] [A-E] [A-E]$').all(), "Invalid prediction format!"

report_df = pd.DataFrame([
    {"Model Architecture": "BERT-Base-Uncased Transformer Ranker", "MAP@3": f"{m_bert['map3']:.4f}", "Top-1 Acc": f"{m_bert['top1_accuracy']:.4f}", "Top-3 Acc": f"{m_bert['top3_accuracy']:.4f}", "F1 Score": f"{m_bert['f1_score']:.4f}", "Log Loss": f"{val_loss_score:.4f}"}
])

print("\n" + "="*70)
print("          MODEL COMPARATIVE REPORT FOR REPORT DOCUMENTATION")
print("="*70)
print(report_df.to_string(index=False))
print("="*70 + "\n")

print("Submission Sample:")
submission.head(10)
